In [32]:
!pip install google-adk -q
!pip install litellm -q

In [33]:
%pip install -q google-adk python-dotenv nest_asyncio

In [3]:
%pip install "litellm>=1.84" -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.2/24.2 MB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.3/278.3 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.7/15.7 MB 63.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 5.1 MB/s eta 0:00:00


In [34]:
import os
from google.adk.models.lite_llm import LiteLlm
from google.genai import types
import asyncio
from google.adk.agents import LlmAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService

import warnings
warnings.filterwarnings("ignore")

import logging
logging.basicConfig(level=logging.ERROR)

print("Bibliotecas importadas.")

Bibliotecas importadas.


In [57]:
try:
  from google.colab import userdata
  GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
  print("Chave carregada")
except Exception:
  import getpass
  GOOGLE_API_KEY = getpass.getpass("Cole sua chave aqui: ")

os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "False"
os.environ["OTEL_SDK_DISABLED"] = "true"

print("Configuração concluída.")

Chave carregada
Configuração concluída.


In [58]:
#verificar se a chave está configurada (colab do tutorial)
print("API Keys Set:")
print(
    f"Google API Key set: {'Yes' if os.environ.get('GOOGLE_API_KEY') and os.environ['GOOGLE_API_KEY'] != 'YOUR_GOOGLE_API_KEY' else 'No (REPLACE PLACEHOLDER!)'}"
)
print(
    f"OpenAI API Key set: {'Yes' if os.environ.get('OPENAI_API_KEY') and os.environ['OPENAI_API_KEY'] != 'YOUR_OPENAI_API_KEY' else 'No (REPLACE PLACEHOLDER!)'}"
)
print(
    f"Anthropic API Key set: {'Yes' if os.environ.get('ANTHROPIC_API_KEY') and os.environ['ANTHROPIC_API_KEY'] != 'YOUR_ANTHROPIC_API_KEY' else 'No (REPLACE PLACEHOLDER!)'}"
)



API Keys Set:
Google API Key set: Yes
OpenAI API Key set: No (REPLACE PLACEHOLDER!)
Anthropic API Key set: No (REPLACE PLACEHOLDER!)


In [71]:
#definindo os modelos (ajuda do tutorial)

#gemini flash
MODEL_GEMINI_2_0_FLASH = "gemini-3.6-flash"

#gpt
MODEL_GPT_4O = "openai/gpt-4.1"

#claude sonnet
MODEL_CLAUDE_SONNET = "anthropic/claude-sonnet-4-20250514"

print("\n Ambiente de modelos configurado.")


 Ambiente de modelos configurado.


In [72]:
#configurando a função de mapeamento da cidade

def get_weather(city: str) -> dict:
    """Recupera a previsão do tempo atual para uma cidade específica.

    Args:
        city (str): O nome da cidade (e.g., "São Paulo", "Curitiba", "Goiânia").

    Returns:
        dict: Um dicionário contendo as informações meteorológicas.
              Inclui uma chave de status ('success' ou 'error').
              Se 'success', inclui uma chave 'report' com detalhes do tempo.
              Se 'error', inclui uma chave 'error_message'.
    """
    print(f"--- Ferramenta: get_weather para a cidade: {city} ---")
    city_normalized = city.lower().replace(" ", "")

    # Mock weather data
    mock_weather_db = {
        "saopaulo": {"status": "success", "report": "O tempo em São Paulo está ensolarado com temperatura de 25°C."},
        "curitiba": {"status": "success", "report": "Há algumas nuvens em Curitiba com temperatura de 15°C."},
        "goiania": {"status": "success", "report": "Tempo quente e seco em Goiânia com temperatura de 35°C."},
    }

    if city_normalized in mock_weather_db:
        return mock_weather_db[city_normalized]
    else:
        return {"status": "error", "error_message": f"Desculpe, eu não tenho informações do clima para a cidade '{city}'."}


print(get_weather("Goiania"))
print(get_weather("Paris"))

--- Ferramenta: get_weather para a cidade: Goiania ---
{'status': 'success', 'report': 'Tempo quente e seco em Goiânia com temperatura de 35°C.'}
--- Ferramenta: get_weather para a cidade: Paris ---
{'status': 'error', 'error_message': "Desculpe, eu não tenho informações do clima para a cidade 'Paris'."}


In [73]:
#Definindo o Agente de Tempo
AGENT_MODEL = MODEL_GEMINI_2_0_FLASH # Starting with Gemini

weather_agent = LlmAgent(
    name="weather_agent_v1",
    model=AGENT_MODEL, # Can be a string for Gemini or a LiteLlm object
    description="Fornece informações meteorológicas para cidades específicas.",
    instruction="Você é um agente meteorológico prestativo."
                "Quando o usuário solicitar a previsão de tempo para uma cidade específica, "
                "use a ferramenta 'get_weather' para encontrar as informações. "
                "Se a ferramenta retornar um erro, informe o usuário de forma educada. "
                "Se a ferramenta for bem sucedida, apresente a previsão do tempo de forma clara.",
    tools=[get_weather], # Pass the function directly
)

print(f"Agent '{weather_agent.name}' created using model '{AGENT_MODEL}'.")

Agent 'weather_agent_v1' created using model 'gemini-3.6-flash'.


In [74]:
#configurando o runner e o serviço de sessão

import nest_asyncio
nest_asyncio.apply()

async def setup_runner():
    session_service = InMemorySessionService()

    APP_NAME = "weather_tutorial_app"
    USER_ID = "user_1"
    SESSION_ID = "session_001"

    session = await session_service.create_session(
        app_name=APP_NAME,
        user_id=USER_ID,
        session_id=SESSION_ID
    )
    print(f"Session created: App='{APP_NAME}', User='{USER_ID}', Session='{SESSION_ID}'")

    runner = Runner(
        agent=weather_agent,
        app_name=APP_NAME,
        session_service=session_service
    )
    print(f"Runner created for agent '{runner.agent.name}'.")

    return runner, USER_ID, SESSION_ID

# Criar uma função async para executar o setup
async def run_setup():
    return await setup_runner()

# Executar
runner, USER_ID, SESSION_ID = await run_setup()

Session created: App='weather_tutorial_app', User='user_1', Session='session_001'
Runner created for agent 'weather_agent_v1'.


In [75]:
#Função de interação do agente

from google.genai import types

async def call_agent_async(query: str, runner, user_id, session_id):
    """Envia a pergunta ao agente e imprime a resposta."""
    print(f"\n>>> Query do usuário: {query}")

    content = types.Content(role="user", parts=[types.Part(text=query)])

    final_response_text = "O agente não apresentou uma resposta final."
    async for event in runner.run_async(
        user_id=user_id, session_id=session_id, new_message=content
    ):
        if event.is_final_response():
            if event.content and event.content.parts:
                final_response_text = event.content.parts[0].text
            elif event.actions and event.actions.escalate:
                final_response_text = f"Resposta do agente: {event.error_message or 'Nenhuma mensagem específica.'}"
            break

    print(f"<<< Resposta do agente: {final_response_text}")
    return final_response_text

In [65]:
# Função para testar múltiplas queries
async def run_conversation():
    await call_agent_async(
        "Como está o tempo em Sao Paulo?",
        runner=runner,
        user_id=USER_ID,
        session_id=SESSION_ID,
    )

    await call_agent_async(
        "E sobre Curitiba?", runner=runner, user_id=USER_ID, session_id=SESSION_ID
    )  # Expecting the tool's error message

    await call_agent_async(
        "Me fale sobre o tempo em Goiania",
        runner=runner,
        user_id=USER_ID,
        session_id=SESSION_ID,
    )

    await call_agent_async(
        "E sobre Paris, tem alguma novidade?",
        runner=runner,
        user_id=USER_ID,
        session_id=SESSION_ID,
    )


# Execute the conversation using await in an async context (like Colab/Jupyter)
await run_conversation()



>>> Query do usuário: Como está o tempo em Sao Paulo?
--- Ferramenta: get_weather para a cidade: Sao Paulo ---


ERROR:opentelemetry.context:Failed to detach context
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/opentelemetry/trace/__init__.py", line 608, in use_span
    yield span
  File "/usr/local/lib/python3.13/dist-packages/opentelemetry/trace/__init__.py", line 508, in start_as_current_span
    yield span
  File "/usr/local/lib/python3.13/dist-packages/opentelemetry/trace/__init__.py", line 443, in start_as_current_span
    yield span
  File "/usr/local/lib/python3.13/dist-packages/google/adk/telemetry/_instrumentation.py", line 78, in record_invocation
    yield
  File "/usr/local/lib/python3.13/dist-packages/google/adk/runners.py", line 699, in _run_node_async
    yield event
GeneratorExit

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/opentelemetry/context/__init__.py", line 143, in detach
    _RUNTIME_CONTEXT.detach(token)
    ~~~~~~~~~~~~~~~~~~~

<<< Resposta do agente: O tempo em São Paulo continua ensolarado, com temperatura de 25°C.

>>> Query do usuário: E sobre Curitiba?


ERROR:google_adk.google.adk.workflow._node_runner:Node execution failed with exception
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/google/adk/models/google_llm.py", line 288, in generate_content_async
    response = await self.api_client.aio.models.generate_content(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<3 lines>...
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/google/genai/models.py", line 8711, in generate_content
    return await self._generate_content(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        model=model, contents=contents, config=final_parsed_config
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/google/genai/models.py", line 7185, in _generate_content
    response = await self._api_client.async_request(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        'post', path, request_dict, http_options
    

<<< Resposta do agente: O agente não apresentou uma resposta final.

>>> Query do usuário: Me fale sobre o tempo em Goiania


ERROR:google_adk.google.adk.workflow._node_runner:Node execution failed with exception
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/google/adk/models/google_llm.py", line 288, in generate_content_async
    response = await self.api_client.aio.models.generate_content(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<3 lines>...
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/google/genai/models.py", line 8711, in generate_content
    return await self._generate_content(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        model=model, contents=contents, config=final_parsed_config
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/google/genai/models.py", line 7185, in _generate_content
    response = await self._api_client.async_request(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        'post', path, request_dict, http_options
    

<<< Resposta do agente: O agente não apresentou uma resposta final.

>>> Query do usuário: E sobre Paris, tem alguma novidade?


ERROR:google_adk.google.adk.workflow._node_runner:Node execution failed with exception
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/google/adk/models/google_llm.py", line 288, in generate_content_async
    response = await self.api_client.aio.models.generate_content(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<3 lines>...
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/google/genai/models.py", line 8711, in generate_content
    return await self._generate_content(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        model=model, contents=contents, config=final_parsed_config
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/google/genai/models.py", line 7185, in _generate_content
    response = await self._api_client.async_request(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        'post', path, request_dict, http_options
    

<<< Resposta do agente: O agente não apresentou uma resposta final.
